# Threshold restriction table verification

Run with the **SageMath** Jupyter kernel. The full computation is contained in the single code cell below. This version uses the correct upper-threshold formula `sum_{j=d}^{n-1} binomial(n-1,j)`.

In [3]:

# Threshold restriction table verification for SageMath/Jupyter.
# Single executable cell. Run with a SageMath kernel.
#
# IMPORTANT CORRECTION:
# In the upper-threshold regime, the coordinate restriction H_{1,0}
# gives T_{d,n-1}.  For d>(n+1)/2, this restricted function has degree/threshold
# above the middle of n-1 variables, and its nonlinearity equals its weight:
#
#     NL(T_{d,n-1}) = sum_{j=d}^{n-1} binom(n-1,j).
#
# Thus the correct upper-threshold rnnl_1 formula checked here is
#
#     rnnl_1(T_{d,n}) = sum_{j=d}^{n-1} binom(n-1,j),
#
# not the erroneous expression with n-2.

from sage.all import *

# ============================================================
# Basic Boolean/Walsh machinery
# ============================================================

def bit_weight(x, n):
    return sum((x >> j) & 1 for j in range(n))

def fwht_pm1(values):
    """
    Fast Walsh-Hadamard transform of (-1)^{f(x)}.
    Input: values[x] in {0,1}.
    Output: W_f(a)=sum_x (-1)^{f(x)+a.x}.
    """
    W = [1 - 2*Integer(v) for v in values]
    N = len(W)
    h = 1
    while h < N:
        for start in range(0, N, 2*h):
            for j in range(start, start+h):
                u = W[j]
                v = W[j+h]
                W[j] = u + v
                W[j+h] = u - v
        h *= 2
    return W

def nl_and_maxwalsh(values):
    """
    Nonlinearity of an m-variable Boolean function from its truth table.
    """
    N = Integer(len(values))
    m = N.exact_log(2)
    W = fwht_pm1(values)
    maxW = max(abs(w) for w in W)
    nl = 2**(m-1) - maxW//2
    return Integer(nl), Integer(maxW), W

# ============================================================
# Hyperplane restrictions H_{i,epsilon}
# ============================================================

def restriction_truth_table_H_i_eps(n, d, i, eps):
    """
    Truth table of T_{d,n} restricted to
        H_{i,eps}: x_1+...+x_i=eps.

    Since T_{d,n} is symmetric, every affine hyperplane is equivalent
    under a coordinate permutation to one of these types.  We parametrize
    H_{i,eps} by free variables x_2,...,x_n and solve for x_1.
    """
    assert 1 <= i <= n
    assert eps in [0, 1]

    values = []
    for y in range(2**(n-1)):
        bits = [0]*n

        # x_2,...,x_n are free.
        for t in range(n-1):
            bits[t+1] = (y >> t) & 1

        # x_1 = eps + x_2 + ... + x_i mod 2.
        s = sum(bits[j] for j in range(1, i))
        bits[0] = (eps + s) % 2

        values.append(1 if sum(bits) >= d else 0)

    return values

def type_record(n, d, i, eps):
    vals = restriction_truth_table_H_i_eps(n, d, i, eps)
    nl, maxW, W = nl_and_maxwalsh(vals)
    return {
        "n": Integer(n),
        "d": Integer(d),
        "i": Integer(i),
        "eps": Integer(eps),
        "nl": Integer(nl),
        "maxW": Integer(maxW),
        "spectrum": W,
    }

def all_type_records(n, d):
    return [type_record(n, d, i, eps) for i in range(1, n+1) for eps in [0, 1]]

def rnnl1_by_types(n, d):
    records = all_type_records(n, d)
    r = min(rec["nl"] for rec in records)
    best = [rec for rec in records if rec["nl"] == r]
    return Integer(r), best, records

# ============================================================
# Formulae checked by the code
# ============================================================

def formula_upper_threshold_correct(n, d):
    """
    Correct upper-threshold value:
        rnnl_1(T_{d,n}) = NL(T_{d,n-1})
                         = sum_{j=d}^{n-1} binom(n-1,j),
    in the regime d>(n+1)/2.

    Example: n=8,d=5 gives C(7,5)+C(7,6)+C(7,7)=29.
    """
    return Integer(sum(binomial(n-1, j) for j in range(d, n)))

def formula_odd_majority(n):
    """
    Odd-majority value:
        rnnl_1(T_{(n+1)/2,n}) = 2^{n-2} - binom(n-1,(n-1)/2).
    """
    assert n % 2 == 1
    return Integer(2**(n-2) - binomial(n-1, (n-1)//2))

def classify_row(n, d):
    if n % 2 == 1 and d == (n+1)//2:
        return "odd majority", formula_odd_majority(n)
    if QQ(d) > QQ(n+1)/2:
        return "upper threshold", formula_upper_threshold_correct(n, d)
    return "outside stated theorem", None

# ============================================================
# Pretty printing
# ============================================================

def qlatex(x):
    return latex(QQ(x))

def type_name(rec):
    return "H_{%s,%s}" % (rec["i"], rec["eps"])

def type_names(records):
    return ", ".join(type_name(rec) for rec in records)

def print_type_audit(n, d):
    print("="*88)
    print("Full type audit for n=%s, d=%s" % (n, d))
    print("type        nl        max |Walsh|")
    print("-"*88)
    for rec in all_type_records(n, d):
        print("%-10s  %-8s  %-8s" % (type_name(rec), rec["nl"], rec["maxW"]))
    r, best, _ = rnnl1_by_types(n, d)
    print("-"*88)
    print("rnnl_1 = %s; best types: %s" % (r, type_names(best)))
    print()

def print_selected_rows(rows):
    print("="*88)
    print("Selected threshold-restriction rows")
    print("="*88)
    print("n & d & regime & computed rnnl_1 & formula/status & best hyperplane types \\\\")
    print("\\hline")
    for n, d in rows:
        rnnl, best, _ = rnnl1_by_types(n, d)
        regime, formula = classify_row(n, d)

        if formula is None:
            formula_text = "computed only"
            check_text = "NO FORMULA ASSERTED"
        else:
            assert rnnl == formula, (n, d, regime, rnnl, formula, type_names(best))
            formula_text = "$%s$" % qlatex(formula)
            check_text = "OK"

        row = "%s & %s & %s & $%s$ & %s & $%s$ \\\\  %% %s" % (
            n, d, regime, qlatex(rnnl), formula_text, type_names(best), check_text
        )
        print(row)
    print()

def verify_theorem_regimes(max_n=12):
    """
    Exhaustively verifies the encoded formulas exactly where they are stated:
    odd majority and upper threshold.
    """
    print("="*88)
    print("Formula verification in stated regimes up to n=%s" % max_n)
    print("="*88)

    for n in range(3, max_n+1):
        for d in range(1, n+1):
            regime, formula = classify_row(n, d)
            if formula is None:
                continue

            rnnl, best, _ = rnnl1_by_types(n, d)
            assert rnnl == formula, (n, d, regime, rnnl, formula, type_names(best))
            print("OK: n=%2s d=%2s %-16s rnnl_1=%-8s best=%s" %
                  (n, d, regime, rnnl, type_names(best)))
    print()

def print_all_rows_small(max_n=9):
    print("="*88)
    print("All rows up to n=%s, including outside-theorem rows" % max_n)
    print("="*88)
    print("n  d  regime                  rnnl_1   formula/status       best")
    print("-"*88)

    for n in range(2, max_n+1):
        for d in range(1, n+1):
            rnnl, best, _ = rnnl1_by_types(n, d)
            regime, formula = classify_row(n, d)
            if formula is None:
                fs = "computed only"
            else:
                assert rnnl == formula, (n, d, regime, rnnl, formula, type_names(best))
                fs = str(formula)
            print("%2s %2s %-23s %-8s %-20s %s" %
                  (n, d, regime, rnnl, fs, type_names(best)))
    print()

# ============================================================
# Manuscript/computational rows
# ============================================================

selected_rows = [
    (5, 3),    # odd majority
    (7, 3),    # outside the two stated regimes; computed only
    (7, 4),    # odd majority
    (8, 5),    # upper threshold; correct value is 29
    (9, 4),    # outside the two stated regimes; computed only
    (9, 5),    # odd majority
    (11, 6),   # odd majority
]

print_selected_rows(selected_rows)

# Verifies formulas exactly where they are stated.
verify_theorem_regimes(max_n=12)

# Optional broader diagnostics:
# print_all_rows_small(max_n=9)

# Optional type audits:
# for n, d in selected_rows:
#     print_type_audit(n, d)


Selected threshold-restriction rows
n & d & regime & computed rnnl_1 & formula/status & best hyperplane types \\
\hline
5 & 3 & odd majority & $2$ & $2$ & $H_{2,0}, H_{4,0}$ \\  % OK
7 & 3 & outside stated theorem & $7$ & computed only & $H_{1,1}, H_{7,1}$ \\  % NO FORMULA ASSERTED
7 & 4 & odd majority & $12$ & $12$ & $H_{2,0}, H_{6,1}$ \\  % OK
8 & 5 & upper threshold & $29$ & $29$ & $H_{1,0}, H_{2,0}, H_{7,0}, H_{8,0}$ \\  % OK
9 & 4 & outside stated theorem & $37$ & computed only & $H_{1,1}, H_{9,0}$ \\  % NO FORMULA ASSERTED
9 & 5 & odd majority & $58$ & $58$ & $H_{2,0}, H_{8,0}$ \\  % OK
11 & 6 & odd majority & $260$ & $260$ & $H_{2,0}, H_{10,1}$ \\  % OK

Formula verification in stated regimes up to n=12
OK: n= 3 d= 2 odd majority     rnnl_1=0        best=H_{2,0}, H_{2,1}
OK: n= 3 d= 3 upper threshold  rnnl_1=0        best=H_{1,0}, H_{2,1}, H_{3,0}
OK: n= 4 d= 3 upper threshold  rnnl_1=1        best=H_{1,0}, H_{2,0}, H_{3,0}, H_{4,0}
OK: n= 4 d= 4 upper threshold  rnnl_1=0       